In [45]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from gensim.models import Word2Vec

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
np.set_printoptions(precision=3, suppress=True)

## 1. One Hot Encoding (OHE)

**What it does:**  
OHE creates a vocabulary and represents each word with a vector containing one `1` and zeros elsewhere. It preserves word order but does not capture word meaning.

**Why it was introduced:**  
It converts text into numerical form so that basic mathematical and logical operations can be performed.

**Limitations:**  
The vectors are sparse and high dimensional. OHE does not show similarity between words, so synonyms and antonyms are treated as equally unrelated.

In [46]:
sentence = "he started driving when he was 22 years old"
tokens = sentence.split()

# Build vocabulary (sorted, unique words) -- matches the paper's construction
vocab = sorted(set(tokens))
print("Vocabulary (sorted):", vocab)
print("Vocabulary size |V| =", len(vocab))

# One-hot encode each token position using sklearn's OneHotEncoder
encoder = OneHotEncoder(categories=[vocab], sparse_output=False)
token_array = np.array(tokens).reshape(-1, 1)
ohe_matrix = encoder.fit_transform(token_array)

ohe_df = pd.DataFrame(ohe_matrix.astype(int), columns=vocab, index=tokens)
ohe_df.index.name = "word position"
ohe_df


Vocabulary (sorted): ['22', 'driving', 'he', 'old', 'started', 'was', 'when', 'years']
Vocabulary size |V| = 8


,22,driving,he,old,started,was,when,years
word position,,,,,,,,
he,0,0,1,0,0,0,0,0
started,0,0,0,0,1,0,0,0
driving,0,1,0,0,0,0,0,0
when,0,0,0,0,0,0,1,0
he,0,0,1,0,0,0,0,0
was,0,0,0,0,0,1,0,0
22,1,0,0,0,0,0,0,0
years,0,0,0,0,0,0,0,1
old,0,0,0,1,0,0,0,0


In [47]:
print("Matrix shape (rows = words in sentence, columns = |V|):", ohe_matrix.shape)

v_he = ohe_df.loc["he"].iloc[0].values if False else ohe_matrix[tokens.index("he")]
v_was = ohe_matrix[tokens.index("was")]
print("Dot product between one-hot vectors for 'he' and 'was':", np.dot(v_he, v_was))


Matrix shape (rows = words in sentence, columns = |V|): (9, 8)
Dot product between one-hot vectors for 'he' and 'was': 0.0


**Observation:**  
The matrix has one row per word occurrence and one column per vocabulary word. Each row contains exactly one `1`. The dot product between different words is `0`, confirming that OHE records word identity and position but not semantic similarity.

## 2. Bag of Words (BoW)

**What it does:**  
BoW represents each document as one row, with columns for vocabulary words. The values show either word frequency or presence and absence. This creates a more compact document term matrix than OHE.

**Why it was introduced:**  
It reduces the large matrix created by OHE and allows documents to be compared using measures such as cosine similarity.

**Limitations:**  
BoW ignores word order and relies on exact word matching. Therefore, sentences with the same words in different orders may have the same representation, and synonyms are not recognized as similar.

In [48]:
corpus = [
    "Sam started driving when he was 22 years old.",
    "He was passionate about swimming, and won several competitions.",
    "He didn't like watching television, as he preferred outdoor activities."
]

count_vectorizer = CountVectorizer(lowercase=True)
bow_matrix = count_vectorizer.fit_transform(corpus)

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=count_vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)
print("BoW matrix shape (documents x vocabulary):", bow_df.shape)
bow_df


BoW matrix shape (documents x vocabulary): (3, 24)


,22,about,activities,and,as,competitions,didn,driving,he,like,old,outdoor,passionate,preferred,sam,several,started,swimming,television,was,watching,when,won,years
Doc 1,1,0,0,0,0,0,0,1,1,0,1,0,0,0,1,0,1,0,0,1,0,1,0,1
Doc 2,0,1,0,1,0,1,0,0,1,0,0,0,1,0,0,1,0,1,0,1,0,0,1,0
Doc 3,0,0,1,0,1,0,1,0,2,1,0,1,0,1,0,0,0,0,1,0,1,0,0,0


In [49]:
# documenting the similarity via dot product / cosine similarity, as discussed in the survey
bow_similarity = cosine_similarity(bow_matrix)
sim_df = pd.DataFrame(bow_similarity,
                       index=[f"Doc {i+1}" for i in range(len(corpus))],
                       columns=[f"Doc {i+1}" for i in range(len(corpus))])
print("Cosine similarity between documents (Bag of Words vectors):")
sim_df


Cosine similarity between documents (Bag of Words vectors):


,Doc 1,Doc 2,Doc 3
Doc 1,1.000000,0.222222,0.19245
Doc 2,0.222222,1.000000,0.19245
Doc 3,0.192450,0.192450,1.00000


**Observation:** The matrix is far smaller than the word by vocabulary OHE matrix
(3 rows instead of one row per word), and it computes a similarity score between
documents. However, the similarity scores only reflect vocabulary overlap.
Document 2 and Document 3 share the word "he" and little else, so their similarity is low
even though both describe personal habits, illustrating BoW's inability to capture
semantic relatedness beyond exact word overlap.


## 3. TF-IDF (Term Frequency–Inverse Document Frequency)

**What it does:**  
TF-IDF improves BoW by assigning weights to words based on their frequency in a document and their rarity across the corpus. Frequent words in one document receive higher weights when they are uncommon in other documents.

**Why it was introduced:**  
Plain word frequency can give high importance to common words or words that appear frequently only because a document is long. TF-IDF reduces the weight of common words such as “the” and “and” because they are less useful for distinguishing documents.

**Limitations:**  
Like BoW, TF-IDF relies on exact word matching and does not recognize synonyms. Its vectors also remain high-dimensional and sparse.

In [50]:
tfidf_corpus = [
    "The house needs repair and the repair cost is high.",           # Doc A - repair/cost
    "The neighborhood has good schools and safe commute options.",   # Doc B - community
    "The apartment repair and maintenance labor cost was expensive.",# Doc C - repair/cost
    "Good schools, grocery stores and a safe environment nearby."    # Doc D - community
]

tfidf_vectorizer = TfidfVectorizer(lowercase=True)
tfidf_matrix = tfidf_vectorizer.fit_transform(tfidf_corpus)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=["Doc A", "Doc B", "Doc C", "Doc D"]
)
print("TF-IDF matrix shape:", tfidf_df.shape)
tfidf_df.round(3)


TF-IDF matrix shape: (4, 24)


,and,apartment,commute,cost,environment,expensive,good,grocery,has,high,house,is,labor,maintenance,nearby,needs,neighborhood,options,repair,safe,schools,stores,the,was
Doc A,0.174,0.00,0.000,0.263,0.000,0.00,0.000,0.000,0.000,0.333,0.333,0.333,0.00,0.00,0.000,0.333,0.000,0.000,0.525,0.000,0.000,0.000,0.425,0.00
Doc B,0.204,0.00,0.391,0.000,0.000,0.00,0.308,0.000,0.391,0.000,0.000,0.000,0.00,0.00,0.000,0.000,0.391,0.391,0.000,0.308,0.308,0.000,0.250,0.00
Doc C,0.198,0.38,0.000,0.300,0.000,0.38,0.000,0.000,0.000,0.000,0.000,0.000,0.38,0.38,0.000,0.000,0.000,0.000,0.300,0.000,0.000,0.000,0.243,0.38
Doc D,0.211,0.00,0.000,0.000,0.404,0.00,0.318,0.404,0.000,0.000,0.000,0.000,0.00,0.00,0.404,0.000,0.000,0.000,0.000,0.318,0.318,0.404,0.000,0.00


In [51]:
tfidf_similarity = cosine_similarity(tfidf_matrix)
tfidf_sim_df = pd.DataFrame(tfidf_similarity,
                             index=["Doc A", "Doc B", "Doc C", "Doc D"],
                             columns=["Doc A", "Doc B", "Doc C", "Doc D"])
print("Cosine similarity between documents (TF-IDF vectors):")
tfidf_sim_df.round(3)


Cosine similarity between documents (TF-IDF vectors):


,Doc A,Doc B,Doc C,Doc D
Doc A,1.000,0.142,0.374,0.037
Doc B,0.142,1.000,0.101,0.337
Doc C,0.374,0.101,1.000,0.042
Doc D,0.037,0.337,0.042,1.000


**Observation:** Documents A and C (both about repair costs) receive a noticeably
higher similarity score than any repair document paired with a community document (B or
D), and B and D (both about neighborhood amenities) are also grouped together. This
matches the paper's discussion: TF-IDF gives more weight to words such as "repair",
"cost", "schools" and "safe" that are informative for distinguishing topics, whereas a
plain frequency count (or NTF alone) would have treated all four documents as similarly
related simply because they share common words like "the" and "and".


## 4. Word2Vec

**What it does:**  
Word2Vec is a neural network based static embedding technique. It learns dense, low dimensional vectors by predicting surrounding words. This notebook uses the Skip-gram architecture, which predicts nearby words from a given word.

**Why it was introduced:**  
OHE, BoW, and TF-IDF cannot capture semantic relationships and produce sparse, high-dimensional vectors. Word2Vec represents words in a continuous vector space where similar words have similar vectors. It can also capture relationships through vector operations.

**Limitations:**  
Word2Vec assigns one fixed vector to each word regardless of context. It cannot distinguish different meanings of a word such as “bank” in “financial bank” and “riverbank”. This limitation later motivated contextual embeddings such as ELMo and BERT.

In [52]:
import random
random.seed(42)

royal_templates = [
    "the {a} rules the kingdom",
    "the {a} wears a golden crown",
    "the {a} lives in the royal palace",
    "the {a} sits on the throne",
    "everyone bows to the {a}",
    "the {a} rules with great wisdom",
    "the royal {a} addresses the court",
    "the {a} inherited the throne",
]

common_templates = [
    "the {a} walks to work in the city",
    "the {a} cooks dinner every evening",
    "the {a} drives a car to the office",
    "the {a} reads a book at the library",
    "the {a} plays with the children",
    "the {a} enjoys a cup of coffee",
    "the {a} works hard every single day",
    "the {a} talks with friends at the cafe",
]

sentences_text = []
for w in ["king", "queen"]:
    for t in royal_templates:
        sentences_text.append(t.format(a=w))

for w in ["man", "woman"]:
    for t in common_templates:
        sentences_text.append(t.format(a=w))

# Linking sentences: connect royal terms to their corresponding gendered common term
linking_sentences = [
    "a king is a man who rules the kingdom",
    "a queen is a woman who rules the kingdom",
    "the man became king after the coronation",
    "the woman became queen after the coronation",
    "the young man was crowned king of the kingdom",
    "the young woman was crowned queen of the kingdom",
    "he is a man and he is now the king",
    "she is a woman and she is now the queen",
]
sentences_text += linking_sentences * 8

random.shuffle(sentences_text)
tokenized_sentences = [s.split() for s in sentences_text]
print("Number of training sentences:", len(tokenized_sentences))
print("Example sentences:", sentences_text[:3])


Number of training sentences: 96
Example sentences: ['the young man was crowned king of the kingdom', 'the woman became queen after the coronation', 'he is a man and he is now the king']


In [53]:
w2v_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=50,      # dimensionality of the learned word vectors
    window=5,            # context window size
    min_count=1,         # keep all words (toy corpus is small)
    sg=1,                # sg=1 -> Skip-gram (as described in the survey); sg=0 -> CBOW
    negative=10,         # negative sampling, helps separate unrelated words
    epochs=300,
    seed=42
)

print("Vocabulary size learned by the model:", len(w2v_model.wv.index_to_key))
print("\nVector for 'king' (first 10 of", w2v_model.wv.vector_size, "dimensions):")
print(w2v_model.wv["king"][:10])


Vocabulary size learned by the model: 66

Vector for 'king' (first 10 of 50 dimensions):
[-0.288 -0.059 -0.511  0.103  0.435 -0.274  0.152 -0.044 -0.398 -0.056]


In [54]:
print("Words most similar to 'king':")
for word, score in w2v_model.wv.most_similar("king", topn=5):
    print(f"  {word:10s}  similarity = {score:.3f}")


Words most similar to 'king':
  queen       similarity = 0.743
  the         similarity = 0.570
  court       similarity = 0.522
  he          similarity = 0.491
  is          similarity = 0.486


In [55]:
result = w2v_model.wv.most_similar(positive=["king", "woman"], negative=["man"], topn=3)
print("king - man + woman  =~")
for word, score in result:
    print(f"  {word:10s}  similarity = {score:.3f}")


king - man + woman  =~
  queen       similarity = 0.885
  the         similarity = 0.505
  kingdom     similarity = 0.473


**Observation:** Even with a small, templated toy corpus, "queen" (or a closely
related royal term) tends to appear among the top results for the `king - man + woman`
analogy, which mirrors the paper's description of word vectors capturing analogies
through simple vector arithmetic. In practice, meaningful analogies of this kind require
training on very large corpora (as in the original Word2Vec paper), so results on a tiny
toy corpus are meant to be illustrative of the *mechanism*, not evidence of production
quality embeddings.
